In [19]:
# ==============================================================================
# STEP 1: INSTALL DEPENDENCIES
# ==============================================================================
!pip install -q langgraph pandas gradio pydantic

# ==============================================================================
# STEP 2: COMPLETE SELF-CONTAINED B2B VENDOR RISK AUDITOR
# ==============================================================================
import os
import pandas as pd
import gradio as gr
from typing import TypedDict, List, Dict, Any
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

# 1. Pydantic Models for Structured Output
class SingleRiskFactor(BaseModel):
    category: str = Field(description="Category of risk")
    risk_level: str = Field(description="HIGH, MEDIUM, or LOW")
    score: int = Field(description="Risk score 0 to 100")
    findings: str = Field(description="Summary of identified risks")
    mitigation_required: str = Field(description="Recommended mitigations")

# 2. State Definition
class VendorAuditState(TypedDict, total=False):
    vendor_name: str
    vendor_industry: str
    annual_revenue_usd: float
    years_in_business: int
    compliance_certifications: List[str]
    pending_litigations: int
    data_breach_history_count: int
    primary_hosting_provider: str
    sla_uptime_guarantee: float

    legal_risk: Dict[str, Any]
    financial_risk: Dict[str, Any]
    cyber_risk: Dict[str, Any]
    operational_risk: Dict[str, Any]

    composite_risk_score: float
    audit_decision: str
    executive_summary: str

# 3. Autonomous Rule-Based Reasoning Agents (Zero External API Dependency)
def legal_compliance_agent(state: VendorAuditState) -> Dict[str, Any]:
    lits = state.get('pending_litigations', 0)
    certs = state.get('compliance_certifications', [])

    if lits > 2:
        level, score = "HIGH", 85
        findings = f"Critical exposure: {lits} active pending litigations detected."
        mitigation = "Require legal indemnity clause and litigation hold clearance."
    elif lits > 0:
        level, score = "MEDIUM", 45
        findings = f"Moderate legal risk with {lits} pending litigation."
        mitigation = "Monitor legal proceedings and cap liability in agreement."
    else:
        level, score = "LOW", 15
        findings = "No active litigation. Strong corporate legal standing."
        mitigation = "Standard legal terms apply."

    return {
        "legal_risk": SingleRiskFactor(
            category="Legal & Compliance",
            risk_level=level,
            score=score,
            findings=findings,
            mitigation_required=mitigation
        ).model_dump()
    }

def financial_health_agent(state: VendorAuditState) -> Dict[str, Any]:
    revenue = state.get('annual_revenue_usd', 0)
    years = state.get('years_in_business', 0)

    if years < 2 or revenue < 1000000:
        level, score = "HIGH", 75
        findings = f"Short operating history ({years} yrs) or low revenue (${revenue:,.0f})."
        mitigation = "Mandate escrow agreement and quarterly financial audits."
    elif years < 5 or revenue < 10000000:
        level, score = "MEDIUM", 40
        findings = f"Growing company profile with ${revenue:,.0f} revenue over {years} years."
        mitigation = "Standard commercial payment terms."
    else:
        level, score = "LOW", 10
        findings = f"Established business with ${revenue:,.0f} annual revenue."
        mitigation = "No special financial conditions required."

    return {
        "financial_risk": SingleRiskFactor(
            category="Financial Health",
            risk_level=level,
            score=score,
            findings=findings,
            mitigation_required=mitigation
        ).model_dump()
    }

def cybersecurity_agent(state: VendorAuditState) -> Dict[str, Any]:
    breaches = state.get('data_breach_history_count', 0)
    certs = [c.upper() for c in state.get('compliance_certifications', [])]
    has_soc2 = any("SOC" in c or "ISO" in c for c in certs)

    if breaches > 0 or not has_soc2:
        level, score = "HIGH", 80
        findings = f"Breach history ({breaches} incident) or missing primary security certs (SOC2/ISO)."
        mitigation = "Mandate annual penetration test report and cyber insurance policy."
    else:
        level, score = "LOW", 15
        findings = f"Clean breach record with key certifications ({', '.join(certs)})."
        mitigation = "Standard data privacy & security addendum."

    return {
        "cyber_risk": SingleRiskFactor(
            category="Cybersecurity & Privacy",
            risk_level=level,
            score=score,
            findings=findings,
            mitigation_required=mitigation
        ).model_dump()
    }

def operational_reliability_agent(state: VendorAuditState) -> Dict[str, Any]:
    sla = state.get('sla_uptime_guarantee', 0.0)

    if sla < 99.0:
        level, score = "HIGH", 85
        findings = f"Subpar SLA commitment of {sla}%. Enterprise outage threshold exceeded."
        mitigation = "Require 99.9% uptime commitment with penalty credits."
    elif sla < 99.9:
        level, score = "MEDIUM", 35
        findings = f"SLA guarantee of {sla}% is standard but exposes minor downtime risk."
        mitigation = "Include service credit remedies in contract."
    else:
        level, score = "LOW", 10
        findings = f"High reliability SLA guarantee of {sla}%."
        mitigation = "Standard operational Monitoring."

    return {
        "operational_risk": SingleRiskFactor(
            category="Operational Reliability",
            risk_level=level,
            score=score,
            findings=findings,
            mitigation_required=mitigation
        ).model_dump()
    }

def composite_score_calculator_node(state: VendorAuditState) -> Dict[str, Any]:
    l_score = state['legal_risk']['score']
    f_score = state['financial_risk']['score']
    c_score = state['cyber_risk']['score']
    o_score = state['operational_risk']['score']

    # Weighted calculation
    composite_score = round((c_score * 0.30) + (l_score * 0.30) + (f_score * 0.20) + (o_score * 0.20), 2)

    if composite_score < 30:
        decision = "APPROVED (Low Risk)"
    elif composite_score < 60:
        decision = "CONDITIONALLY APPROVED (Mitigation Required)"
    else:
        decision = "REJECTED (High Risk Threshold Exceeded)"

    return {"composite_risk_score": composite_score, "audit_decision": decision}

def executive_report_agent(state: VendorAuditState) -> Dict[str, Any]:
    summary = (
        f"An automated multi-agent risk assessment was conducted for '{state.get('vendor_name')}'. "
        f"The composite procurement risk score is {state.get('composite_risk_score')}/100, resulting in an "
        f"audit outcome of {state.get('audit_decision')}.\n\n"
        f"Key Risk Factors:\n"
        f"- Legal: {state['legal_risk']['findings']}\n"
        f"- Financial: {state['financial_risk']['findings']}\n"
        f"- Cybersecurity: {state['cyber_risk']['findings']}\n"
        f"- Operational: {state['operational_risk']['findings']}"
    )
    return {"executive_summary": summary}

# 4. Build LangGraph Workflow
builder = StateGraph(VendorAuditState)
builder.add_node("legal_agent", legal_compliance_agent)
builder.add_node("financial_agent", financial_health_agent)
builder.add_node("cyber_agent", cybersecurity_agent)
builder.add_node("operational_agent", operational_reliability_agent)
builder.add_node("calculator", composite_score_calculator_node)
builder.add_node("executive_agent", executive_report_agent)

builder.set_entry_point("legal_agent")
builder.add_edge("legal_agent", "financial_agent")
builder.add_edge("financial_agent", "cyber_agent")
builder.add_edge("cyber_agent", "operational_agent")
builder.add_edge("operational_agent", "calculator")
builder.add_edge("calculator", "executive_agent")
builder.add_edge("executive_agent", END)

audit_app = builder.compile()

# 5. Gradio Interface
def run_audit_interface(vendor_name, industry, revenue, years_in_biz, certs_str, litigations, breaches, hosting, sla):
    try:
        certs_list = [c.strip() for c in str(certs_str).split(",") if c.strip()]

        initial_state = {
            "vendor_name": str(vendor_name),
            "vendor_industry": str(industry),
            "annual_revenue_usd": float(revenue),
            "years_in_business": int(years_in_biz),
            "compliance_certifications": certs_list,
            "pending_litigations": int(litigations),
            "data_breach_history_count": int(breaches),
            "primary_hosting_provider": str(hosting),
            "sla_uptime_guarantee": float(sla)
        }

        result = audit_app.invoke(initial_state)

        scorecard_df = pd.DataFrame([
            {"Category": "Legal & Compliance", "Level": result['legal_risk']['risk_level'], "Score": result['legal_risk']['score'], "Key Finding": result['legal_risk']['findings']},
            {"Category": "Financial Health", "Level": result['financial_risk']['risk_level'], "Score": result['financial_risk']['score'], "Key Finding": result['financial_risk']['findings']},
            {"Category": "Cybersecurity & Privacy", "Level": result['cyber_risk']['risk_level'], "Score": result['cyber_risk']['score'], "Key Finding": result['cyber_risk']['findings']},
            {"Category": "Operational Reliability", "Level": result['operational_risk']['risk_level'], "Score": result['operational_risk']['score'], "Key Finding": result['operational_risk']['findings']},
        ])

        summary_text = f"### Overall Status: {result['audit_decision']}\n"
        summary_text += f"**Composite Risk Score:** {result['composite_risk_score']} / 100\n\n"
        summary_text += f"### Executive Summary:\n{result['executive_summary']}"

        return summary_text, scorecard_df

    except Exception as e:
        return f"❌ Error during graph execution:\n`{str(e)}`", pd.DataFrame()

# 6. UI Layout
with gr.Blocks(title="B2B Vendor Risk Auditor") as demo:
    gr.Markdown("# 🛡️ Autonomous B2B Vendor Risk Auditor")
    with gr.Row():
        with gr.Column():
            v_name = gr.Textbox(value="CloudTech Analytics Inc.", label="Vendor Name")
            v_ind = gr.Dropdown(choices=["SaaS / Software", "Fintech", "Healthcare"], value="SaaS / Software", label="Industry")
            v_rev = gr.Number(value=4500000, label="Annual Revenue ($ USD)")
            v_years = gr.Number(value=3, label="Years in Business")
            v_certs = gr.Textbox(value="ISO27001, SOC2 Type II", label="Certifications")
            v_lit = gr.Number(value=1, label="Pending Litigations")
            v_breach = gr.Number(value=0, label="Past Data Breaches")
            v_host = gr.Textbox(value="AWS US-East-1", label="Hosting Provider")
            v_sla = gr.Number(value=99.5, label="SLA Uptime (%)")
            btn = gr.Button("🚀 Run Multi-Agent Risk Audit", variant="primary")

        with gr.Column():
            output_summary = gr.Markdown()
            output_table = gr.Dataframe(label="Sub-Audit Scorecard Breakdown")

    btn.click(
        fn=run_audit_interface,
        inputs=[v_name, v_ind, v_rev, v_years, v_certs, v_lit, v_breach, v_host, v_sla],
        outputs=[output_summary, output_table]
    )

demo.launch(share=True, debug=False, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7e9da80e3e50eca54c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
